# LinQ Safety: быстрый эксперимент на двух T4

Все пять задач: три YOLO12m и два ResNet50. Это короткое
дообучение с ограниченным бюджетом, а не эквивалент 150 эпох.
Люди и каски продолжаются из last.pt сохранённой версии 361611151; без них запуск
останавливается. Сохраняются optimizer и номер эпохи, ограничивается дополнительное
время. Жилеты и оба классификатора в прежнем запуске ещё не обучались: они начинают
с pretrained с замороженными ранними слоями.
Ожидаемое время 60-90 минут, общий бюджет 110 минут плюс ожидание ресурсов Kaggle.
Приватный расширенный корпус остаётся целым. Используется фиксированная выборка,
сохранены исходные train/val/test и группы. Метрики относятся к выбранной части test.
Расширенную проверку на всём held-out и целевых видео нужно выполнить отдельно.
Оценка >0.95 не гарантирована. Результаты не устанавливаются в сайт автоматически.
Добавьте приватный Dataset LinQ PPE Expanded 2026, GPU T4 x2 и Internet.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','--quiet',
                'ultralytics==8.4.88','onnx==1.19.1','protobuf==5.29.5','onnxruntime==1.29.0',
                'gdown==5.2.0','pyyaml==6.0.3'],check=True)
from pathlib import Path
import json, shutil, zipfile, numpy as np, torch
WORK=Path('/kaggle/working'); WORK.mkdir(exist_ok=True)
assert torch.cuda.is_available(), 'Выберите GPU в Settings -> Accelerator и перезапустите сессию.'
import platform, importlib.metadata as metadata
print({'python':platform.python_version(),'torch':torch.__version__,'onnx':metadata.version('onnx'),
       'gpus':[{'name':torch.cuda.get_device_name(i),'GiB':torch.cuda.get_device_properties(i).total_memory/2**30}
               for i in range(torch.cuda.device_count())]})

In [ ]:
SCRIPTS=WORK/'scripts'; SCRIPTS.mkdir(exist_ok=True)
(SCRIPTS/'download_ppe_data.py').write_text('"""Download public training sources; preserve provenance and actual counts.\n\nSH17 is optional (about 14 GB upstream). No credentials are stored. Archives\nare data only; never run Python from a dataset archive.\n"""\nimport argparse\nimport concurrent.futures\nimport hashlib\nimport json\nfrom pathlib import Path\nimport urllib.request\nimport zipfile\n\nSOURCES = {\n    \'construction\': {\'url\':\'https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip\',\n                     \'license\':\'AGPL-3.0\',\'page\':\'https://docs.ultralytics.com/datasets/detect/construction-ppe\'},\n    \'site_safety\': {\'url\':\'https://www.kaggle.com/api/v1/datasets/download/snehilsanyal/construction-site-safety-image-dataset-roboflow\',\n                    \'license\':\'CC BY 4.0 (source declaration)\',\'page\':\'https://www.kaggle.com/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow\'},\n    \'shwd\': {\'url\':\'https://drive.google.com/uc?id=1qWm7rrwvjAWs1slymbrLaCf7Q-wnGLEX\',\n             \'license\':\'MIT repository; includes SCUT-HEAD-derived images; retain source notices\',\n             \'page\':\'https://github.com/njvisionpower/Safety-Helmet-Wearing-Dataset\'},\n    \'sh17\': {\'url\':\'https://www.kaggle.com/api/v1/datasets/download/mugheesahmad/sh17-dataset-for-ppe-detection\',\n             \'license\':\'CC BY-NC-SA 4.0\',\'page\':\'https://github.com/ahmadmughees/SH17dataset\'},\n    \'hardhat_vest\': {\'url\':\'https://www.kaggle.com/api/v1/datasets/download/muhammetzahitaydn/hardhat-vest-dataset-v3\',\n                     \'license\':\'Check archive source notices; Kaggle declaration alone is not a provenance audit\',\n                     \'page\':\'https://www.kaggle.com/datasets/muhammetzahitaydn/hardhat-vest-dataset-v3\'},\n}\nEXPECTED_SHA256={\n    \'construction\':\'bef8dcb599aa4e9d9f5e602cb6fa7143d3c84d7f6a0ff40463d7f2a4c2632ccc\',\n    \'site_safety\':\'f1acf02c7116a09cdbb2b3ace1ee9b597977c0eb0bd0025e8949f81091fb2a3d\',\n    \'shwd\':\'2f6c5c347b778271bc8b27c2d61b731fc5fb8cdb4f45fce583dd2f82e6a46401\',\n    \'hardhat_vest\':\'61048c7f9a52c8672e1989e338e61781515056ae3809c2fa7a40477ddf20afaf\',\n}\n\n\ndef download_source(name,root):\n    spec=SOURCES[name];root=Path(root);root.mkdir(parents=True,exist_ok=True)\n    target=root/(name+\'.zip\');partial=target.with_suffix(\'.zip.part\')\n    if not target.exists():\n        if name==\'shwd\':\n            import gdown\n            if not gdown.download(spec[\'url\'],str(partial),quiet=True):raise RuntimeError(\'SHWD download failed\')\n        else:\n            offset=partial.stat().st_size if partial.exists() else 0\n            headers={\'User-Agent\':\'LinQSafety dataset preparation/1.0\'}\n            if offset:headers[\'Range\']=f\'bytes={offset}-\'\n            request=urllib.request.Request(spec[\'url\'],headers=headers)\n            with urllib.request.urlopen(request,timeout=120) as response:\n                mode=\'ab\' if response.status==206 and offset else \'wb\'\n                with partial.open(mode) as stream:\n                    while chunk:=response.read(1024*1024):stream.write(chunk)\n        if not zipfile.is_zipfile(partial):raise ValueError(f\'{name}: response is not a ZIP dataset\')\n        partial.replace(target)\n    digest=hashlib.sha256()\n    with target.open(\'rb\') as stream:\n        while chunk:=stream.read(8*1024*1024):digest.update(chunk)\n    if name in EXPECTED_SHA256 and digest.hexdigest()!=EXPECTED_SHA256[name]:\n        raise ValueError(f\'{name}: source archive changed; review its version before using new data\')\n    folder=root/name;marker=folder/\'.extracted\'\n    if not marker.exists():\n        folder.mkdir(exist_ok=True)\n        with zipfile.ZipFile(target) as archive:\n            for item in archive.infolist():\n                path=(folder/item.filename).resolve()\n                if not path.is_relative_to(folder.resolve()):raise ValueError(\'Unsafe archive path\')\n            archive.extractall(folder)\n        marker.write_text(\'ok\',encoding=\'utf-8\')\n    report={\'source\':name,**spec,\'archive_sha256\':digest.hexdigest(),\'bytes\':target.stat().st_size,\n            \'images\':sum(p.suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\'} for p in folder.rglob(\'*\')),\n            \'status\':\'downloaded\',\'folder\':name}\n    (root/(name+\'-source.json\')).write_text(json.dumps(report,indent=2)+\'\\n\',encoding=\'utf-8\')\n    print(json.dumps(report),flush=True);return report\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--sources\',nargs=\'+\',choices=SOURCES,default=[\'construction\',\'site_safety\',\'shwd\'])\n    p.add_argument(\'--root\',default=\'data/datasets/ppe-sources\');args=p.parse_args()\n    failures=[]\n    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:\n        futures={pool.submit(download_source,s,args.root):s for s in args.sources}\n        for f,name in [(f,futures[f]) for f in concurrent.futures.as_completed(futures)]:\n            try:f.result()\n            except Exception as exc:failures.append({\'source\':name,\'status\':\'failed\',\'error\':str(exc)})\n    Path(args.root,\'download-failures.json\').write_text(json.dumps(failures,indent=2),encoding=\'utf-8\')\n    return bool(failures)\n\n\nif __name__==\'__main__\':raise SystemExit(main())\n',encoding='utf-8')
(SCRIPTS/'prepare_ppe_data.py').write_text('"""Build separate, reproducible tasks without treating unlabelled PPE as absent.\n\nImages remain in ignored source folders. Prepared datasets contain real files\nand source/group manifests. Source test/validation images never enter training.\n"""\nimport argparse\nfrom collections import Counter,defaultdict\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nimport xml.etree.ElementTree as ET\n\nimport cv2\nimport numpy as np\nimport yaml\n\nSITE_NAMES=[\'Hardhat\',\'Mask\',\'NO-Hardhat\',\'NO-Mask\',\'NO-Safety Vest\',\'Person\',\n            \'Safety Cone\',\'Safety Vest\',\'machinery\',\'vehicle\']\nALIASES={\'helmet\':\'hardhat\',\'no_helmet\':\'no_hardhat\',\'bare_head\':\'no_hardhat\',\n         \'safety_vest\':\'vest\',\'no_safety_vest\':\'no_vest\'}\nTASKS={\'people\':[\'person\'],\'head_ppe\':[\'hardhat\',\'no_hardhat\'],\'vest_ppe\':[\'vest\',\'no_vest\']}\n\n\ndef crop_quality(crop):\n    """Conservative classifier visibility gate; this does not verify labels."""\n    h,w=crop.shape[:2]\n    if min(h,w)<24 or h*w<1024:return \'too_small\'\n    if not .2<=w/h<=4:return \'extreme_aspect\'\n    if cv2.Laplacian(cv2.cvtColor(crop,cv2.COLOR_BGR2GRAY),cv2.CV_64F).var()<15:\n        return \'blurred\'\n    return None\n\n\ndef normalized(s):\n    s=re.sub(\'[^a-z0-9]+\',\'_\',s.lower()).strip(\'_\')\n    return ALIASES.get(s,s)\n\n\ndef source_split(path):\n    parts=set(path.parts)\n    return \'test\' if \'test\' in parts else \'val\' if parts & {\'val\',\'valid\',\'validation\'} else \'train\'\n\n\ndef original_group(path):\n    # Roboflow augmentation variants must share a split, even with different pixels.\n    return re.sub(r\'(?:_jpg|_png)?\\.rf\\.[a-f0-9]+$\',\'\',path.stem,flags=re.I)\n\n\ndef records(root):\n    for source in sorted(p for p in Path(root).iterdir() if p.is_dir()):\n        if not (source/\'.extracted\').exists():raise ValueError(f\'Source extraction incomplete: {source}\')\n        configs=list(source.rglob(\'*.yaml\'))+list(source.rglob(\'*.yml\'))\n        names=None\n        for cfg in configs:\n            parsed=yaml.safe_load(cfg.read_text(encoding=\'utf-8\'))\n            if isinstance(parsed,dict) and parsed.get(\'names\'):\n                n=parsed[\'names\'];names={int(k):v for k,v in n.items()} if isinstance(n,dict) else dict(enumerate(n));break\n        if names is None and source.name==\'site_safety\':names=dict(enumerate(SITE_NAMES))\n        if names is None:\n            class_files=list(source.rglob(\'classes.txt\'))\n            if class_files:names=dict(enumerate(line.strip() for line in class_files[0].read_text().splitlines() if line.strip()))\n        annotations=list(source.rglob(\'Annotations\'))\n        xmls={p.stem:p for folder in annotations for p in folder.glob(\'*.xml\')}\n        native_splits={}\n        for split in [\'train\',\'val\',\'test\']:\n            for split_file in source.rglob(split+\'.txt\'):\n                if split_file.parent.name==\'Main\':\n                    for line in split_file.read_text().splitlines():\n                        if line.strip():native_splits[line.split()[0]]=split\n        for image in sorted(p for p in source.rglob(\'*\') if p.suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\',\'.webp\'}):\n            labels=[]\n            if xmls:\n                xml=xmls.get(image.stem)\n                if xml is None:continue\n                node=ET.parse(xml).getroot();width=float(node.findtext(\'size/width\'));height=float(node.findtext(\'size/height\'))\n                for obj in node.findall(\'object\'):\n                    label=obj.findtext(\'name\');box=obj.find(\'bndbox\')\n                    # SHWD\'s "person" is an uncovered HEAD, not a person box.\n                    label=\'hardhat\' if label==\'hat\' else \'no_hardhat\' if label==\'person\' else normalized(label)\n                    a,b,c,d=[float(box.findtext(k)) for k in [\'xmin\',\'ymin\',\'xmax\',\'ymax\']]\n                    labels.append([label,(a+c)/2/width,(b+d)/2/height,(c-a)/width,(d-b)/height])\n                complete={\'head_ppe\'}\n            else:\n                if names is None:continue\n                relative=image.relative_to(source)\n                parts=list(relative.parts)\n                if \'images\' not in parts:continue\n                parts[parts.index(\'images\')]=\'labels\'\n                label_path=(source/Path(*parts)).with_suffix(\'.txt\')\n                if not label_path.exists():continue\n                for line in label_path.read_text().splitlines():\n                    fields=line.split()\n                    if len(fields)!=5:raise ValueError(f\'Expected detection boxes, not segmentation: {label_path}\')\n                    label=normalized(names[int(fields[0])]);box=list(map(float,fields[1:]))\n                    if not all(np.isfinite(box)) or not all(0<=v<=1 for v in box) or min(box[2:])<=0:\n                        raise ValueError(f\'Invalid bbox: {label_path}\')\n                    labels.append([label,*box])\n                source_classes={normalized(n) for n in names.values()}\n                complete={task for task,classes in TASKS.items() if set(classes)<=source_classes}\n                # This aggregate mixes several annotation schemes. Use only\n                # explicitly labelled positive crops until completeness and the\n                # meaning of head/person have been reviewed for each subsource.\n                if source.name==\'hardhat_vest\':complete=set()\n            yield {\'path\':str(image.resolve()),\'source\':source.name,\'split\':native_splits.get(image.stem,source_split(image)),\n                   \'group\':original_group(image),\'labels\':labels,\'complete_tasks\':sorted(complete)}\n\n\ndef prepare(root,output,summary_path,extra_roots=()):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    if any(output.iterdir()):\n        raise ValueError(\'Output is not empty; choose a new output to avoid stale files\')\n    manifest=[];hashes={};groups={};buckets=defaultdict(list);duplicates=0;unusable=0\n    # Read held-out partitions first: duplicates and related variants in train\n    # inherit the held-out split. This keeps source test images out of train.\n    raw=[rec for source_root in [root,*extra_roots] for rec in records(source_root)]\n    raw.sort(key=lambda r:({\'test\':0,\'val\':1,\'train\':2}[r[\'split\']],-len(r[\'complete_tasks\']),r[\'path\']))\n    print(f\'Indexed {len(raw)} annotated source images\',flush=True)\n    for number,record in enumerate(raw):\n        if number%1000==0:print(f\'Deduplicating {number}/{len(raw)}\',flush=True)\n        if not record[\'complete_tasks\'] and not any(v[0] in {\'hardhat\',\'vest\'} for v in record[\'labels\']):\n            unusable+=1;continue\n        path=Path(record[\'path\']);data=path.read_bytes();digest=hashlib.sha256(data).hexdigest()\n        image=cv2.imdecode(np.frombuffer(data,np.uint8),cv2.IMREAD_COLOR)\n        if image is None:raise ValueError(f\'Unreadable image: {path}\')\n        gray=cv2.resize(cv2.cvtColor(image,cv2.COLOR_BGR2GRAY),(32,32)).astype(np.float32)\n        low=cv2.dct(gray)[:8,:8].ravel();bits=low>np.median(low[1:])\n        phash=sum(int(v)<<i for i,v in enumerate(bits))\n        candidates=set(i for k in range(8) for i in buckets[(k,(phash>>(k*8))&255)])\n        near=next((i for i in candidates if (manifest[i][\'phash\']^phash).bit_count()<=2),None)\n        duplicate=hashes.get(digest,near)\n        if duplicate is not None:\n            duplicates+=1\n            previous=manifest[duplicate]\n            groups[(record[\'source\'],record[\'group\'])]=previous[\'split\']\n            if len(record[\'complete_tasks\'])>len(previous[\'complete_tasks\']):\n                record.update(split=previous[\'split\'],sha256=digest,phash=phash,width=image.shape[1],height=image.shape[0])\n                manifest[duplicate]=record\n            hashes[digest]=duplicate\n            continue\n        # Group keys may over-group coincident source names; that is conservative\n        # for leakage. Exact/perceptual matches are handled globally above.\n        group=(record[\'source\'],record[\'group\'])\n        split=groups.setdefault(group,record[\'split\'])\n        if split==\'train\':\n            bucket=int(hashlib.sha256(\'|\'.join(group).encode()).hexdigest()[:8],16)%100\n            split=\'val\' if bucket<10 else \'test\' if bucket<20 else \'train\'\n        record.update(split=split,sha256=digest,phash=phash,width=image.shape[1],height=image.shape[0])\n        hashes[digest]=len(manifest)\n        for k in range(8):buckets[(k,(phash>>(k*8))&255)].append(len(manifest))\n        manifest.append(record)\n    rank={\'test\':0,\'val\':1,\'train\':2}\n    for rec in manifest:\n        group_split=groups.get((rec[\'source\'],rec[\'group\']),rec[\'split\'])\n        if rank[group_split]<rank[rec[\'split\']]:rec[\'split\']=group_split\n    task_counts={};crop_counts=defaultdict(Counter)\n    for task,classes in TASKS.items():\n        print(f\'Writing {task}\',flush=True)\n        counts=Counter();folder=output/task\n        for rec in manifest:\n            name=rec[\'sha256\'];split=rec[\'split\'];src=Path(rec[\'path\'])\n            selected=[v for v in rec[\'labels\'] if v[0] in classes]\n            if task in rec[\'complete_tasks\']:\n                im=folder/\'images\'/split/(name+src.suffix.lower());lab=folder/\'labels\'/split/(name+\'.txt\')\n                im.parent.mkdir(parents=True,exist_ok=True);lab.parent.mkdir(parents=True,exist_ok=True)\n                if not im.exists():shutil.copyfile(src,im)\n                lab.write_text(\'\\n\'.join(str(classes.index(v[0]))+\' \'+\' \'.join(map(str,v[1:])) for v in selected)+\'\\n\',encoding=\'utf-8\')\n                counts[split]+=1\n            if task in {\'head_ppe\',\'vest_ppe\'} and selected:\n                frame=cv2.imread(str(src));h,w=frame.shape[:2]\n                for i,(label,cx,cy,bw,bh) in enumerate(selected):\n                    # Context is needed around a helmet box; no synthetic\n                    # "unknown" examples or missing-annotation negatives.\n                    x1,y1=max(0,int((cx-bw*.65)*w)),max(0,int((cy-bh*.65)*h))\n                    x2,y2=min(w,int((cx+bw*.65)*w)),min(h,int((cy+bh*.65)*h))\n                    if min(x2-x1,y2-y1)<16:continue\n                    dest=output/(task+\'_classifier\')/split/label/(name+f\'-{i}.jpg\')\n                    dest.parent.mkdir(parents=True,exist_ok=True)\n                    crop=frame[y1:y2,x1:x2]\n                    if crop_quality(crop):continue\n                    cv2.imwrite(str(dest),crop);crop_counts[task][split+\':\'+label]+=1\n        folder.mkdir(parents=True,exist_ok=True)\n        config={\'path\':str(folder.resolve()),\'train\':\'images/train\',\'val\':\'images/val\',\'test\':\'images/test\',\'names\':dict(enumerate(classes))}\n        (folder/\'data.yaml\').write_text(yaml.safe_dump(config,sort_keys=False),encoding=\'utf-8\')\n        task_counts[task]=dict(counts)\n    with (output/\'manifest.jsonl\').open(\'w\',encoding=\'utf-8\') as f:\n        for rec in manifest:f.write(json.dumps(rec)+\'\\n\')\n    summary={\'raw_images\':len(raw),\'unique_images\':len(manifest),\'duplicates_removed\':duplicates,\n             \'excluded_incomplete_annotation_images\':unusable,\n             \'sources\':dict(Counter(r[\'source\'] for r in manifest)),\'splits\':dict(Counter(r[\'split\'] for r in manifest)),\n             \'tasks\':task_counts,\'classification_crops\':{k:dict(v) for k,v in crop_counts.items()},\n             \'baseline_construction_train_images\':1132,\'upstream_claimed_train_images\':68253,\n             \'upstream_4x_train_target\':273012,\'actual_unique_train_images\':sum(r[\'split\']==\'train\' for r in manifest),\n             \'upstream_4x_reached\':sum(r[\'split\']==\'train\' for r in manifest)>=273012,\n             \'limitations\':[\'Source group metadata may be incomplete; camera-disjoint independence is not established.\',\n                            \'No unobservable-class crops are fabricated; manual visibility labels remain required.\',\n                            \'Head and torso classifier crops need human label/geometry review before a full training run.\']}\n    Path(summary_path).parent.mkdir(parents=True,exist_ok=True)\n    Path(summary_path).write_text(json.dumps(summary,indent=2)+\'\\n\',encoding=\'utf-8\')\n    try:\n        from scripts.finalize_ppe_data import finalize\n    except ModuleNotFoundError:\n        from finalize_ppe_data import finalize\n    summary=finalize(output,summary_path,output.parent/\'ppe-excluded-manifest.jsonl\')\n    print(json.dumps(summary,indent=2));return summary\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--root\',default=\'data/datasets/ppe-sources\')\n    p.add_argument(\'--extra-root\',action=\'append\',default=[])\n    p.add_argument(\'--output\',default=\'data/datasets/ppe-prepared\');p.add_argument(\'--summary\',default=\'docs/ppe-dataset-summary.json\')\n    a=p.parse_args();prepare(a.root,a.output,a.summary,a.extra_root)\n',encoding='utf-8')
(SCRIPTS/'finalize_ppe_data.py').write_text('"""Keep only source images that contribute detector examples or usable crops."""\nimport argparse\nfrom collections import Counter\nimport json\nfrom pathlib import Path\n\n\ndef finalize(data,summary_path,excluded):\n    data=Path(data);summary_path=Path(summary_path);excluded=Path(excluded)\n    summary=json.loads(summary_path.read_text());manifest=data/\'manifest.jsonl\'\n    rows=[json.loads(line) for line in manifest.read_text().splitlines()]\n    used={p.stem for task in [\'people\',\'head_ppe\',\'vest_ppe\'] for p in (data/task/\'images\').rglob(\'*\') if p.is_file()}\n    used.update(p.stem[:64] for task in [\'head_ppe_classifier\',\'vest_ppe_classifier\'] for p in (data/task).rglob(\'*.jpg\'))\n    kept=[r for r in rows if r[\'sha256\'] in used];removed=[r for r in rows if r[\'sha256\'] not in used]\n    summary.setdefault(\'unique_images_before_crop_filter\',summary[\'unique_images\'])\n    summary.setdefault(\'splits_before_crop_filter\',summary[\'splits\'])\n    summary[\'unique_images\']=len(kept)\n    summary[\'excluded_images_without_usable_crops\']=summary[\'unique_images_before_crop_filter\']-len(kept)\n    summary[\'sources\']=dict(Counter(r[\'source\'] for r in kept))\n    summary[\'splits\']=dict(Counter(r[\'split\'] for r in kept))\n    summary[\'actual_unique_train_images\']=summary[\'splits\'].get(\'train\',0)\n    summary[\'upstream_4x_reached\']=summary[\'actual_unique_train_images\']>=summary[\'upstream_4x_train_target\']\n    if removed:\n        excluded.parent.mkdir(parents=True,exist_ok=True)\n        with excluded.open(\'a\',encoding=\'utf-8\') as f:\n            for row in removed:f.write(json.dumps(row)+\'\\n\')\n        temporary=manifest.with_suffix(\'.tmp\')\n        with temporary.open(\'w\',encoding=\'utf-8\') as f:\n            for row in kept:f.write(json.dumps(row)+\'\\n\')\n        temporary.replace(manifest)\n    summary_path.write_text(json.dumps(summary,indent=2)+\'\\n\')\n    print(\'Contributing unique images:\',len(kept),\'train:\',summary[\'actual_unique_train_images\'],flush=True)\n    return summary\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',default=\'data/datasets/ppe-expanded\')\n    p.add_argument(\'--summary\',default=\'docs/ppe-dataset-summary.json\')\n    p.add_argument(\'--excluded\',default=\'data/datasets/ppe-rejected-crops/unused-manifest.jsonl\')\n    a=p.parse_args();finalize(a.data,a.summary,a.excluded)\n',encoding='utf-8')
(SCRIPTS/'train_ppe.py').write_text('"""Train one PPE task; test is evaluated only after validation selects weights."""\nimport argparse\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\nfrom training_checkpoint import atomic_copy,atomic_torch_save\n\n# Sampled train-only QC found a vertically mirrored border labelled as helmet\n# although that crop contains only chin/shoulder. Keep raw data immutable and\n# exclude all classifier crops from this image (no complete detector task).\nQC_EXCLUDED_IMAGE_SHA256={\'8ab6abebd308d47cead1c3b6403e4bc767c288766d4113dde197c3609373a0c3\'}\n\n\ndef valid_classifier_file(path):\n    p=Path(path)\n    return p.suffix.lower() in {\'.jpg\',\'.jpeg\',\'.png\'} and p.stem[:64] not in QC_EXCLUDED_IMAGE_SHA256\n\n\ndef train_detector(args):\n    from ultralytics import YOLO,settings\n    # Disable optional account integrations only in this process. Do not write\n    # the user\'s persistent Ultralytics settings on a local training smoke.\n    dict.update(settings,{key:False for key in [\'wandb\',\'mlflow\',\'comet\',\'clearml\',\'neptune\',\'raytune\',\'hub\'] if key in settings})\n    import yaml\n    output=Path(args.output);output.mkdir(parents=True,exist_ok=True)\n    config=yaml.safe_load((Path(args.data)/args.task/\'data.yaml\').read_text())\n    config[\'path\']=str((Path(args.data)/args.task).resolve())\n    data_file=output/(args.task+\'-data.yaml\');data_file.write_text(yaml.safe_dump(config),encoding=\'utf-8\')\n    last=output/args.task/\'weights/last.pt\'\n    continuation=None\n    if args.continue_checkpoint:\n        import torch,shutil\n        source=Path(args.continue_checkpoint)\n        # A retry must prefer this experiment\'s optimizer-preserving checkpoint.\n        local_resume=last.with_name(\'resume-last.pt\')\n        if args.resume and local_resume.exists():source=local_resume\n        # These are our own saved Kaggle checkpoints, not arbitrary uploaded pickle files.\n        state=torch.load(source,map_location=\'cpu\',weights_only=False)\n        if state.get(\'epoch\',-1)<0 or not state.get(\'optimizer\'):\n            raise ValueError(\'Exact continuation requires an unfinished checkpoint with optimizer state\')\n        if args.freeze:raise ValueError(\'Changing frozen parameter groups is incompatible with exact optimizer continuation\')\n        continuation={\'source_sha256\':hashlib.sha256(source.read_bytes()).hexdigest(),\n                      \'previous_completed_epochs\':int(state[\'epoch\'])+1,\'optimizer_restored\':True}\n        groups=state[\'optimizer\'][\'param_groups\']\n        optimizer_name=\'MuSGD\' if any(g.get(\'use_muon\') for g in groups) else (\'AdamW\' if \'betas\' in groups[0] else \'SGD\')\n        # auto chooses a different optimizer when a shorter schedule drops below\n        # 10k iterations. Pin the saved optimizer family before restoring groups.\n        state[\'train_args\'].update(project=str(output.resolve()),name=args.task,data=str(data_file.resolve()),\n            save_dir=str((output/args.task).resolve()),epochs=args.epochs,optimizer=optimizer_name)\n        continuation[\'optimizer_family\']=optimizer_name\n        last.parent.mkdir(parents=True,exist_ok=True)\n        atomic_torch_save(state,last)\n        old_best=source.parent/\'best.pt\'\n        if old_best.exists() and old_best.resolve()!=(last.parent/\'best.pt\').resolve():\n            atomic_copy(old_best,last.parent/\'best.pt\')\n        history=source.parent.parent/\'results.csv\'\n        if history.exists():\n            rows=history.read_text().splitlines()[:int(state[\'epoch\'])+2]\n            (output/args.task/\'results.csv\').write_text(\'\\n\'.join(rows)+\'\\n\')\n        del state\n        args.resume=True\n    model=YOLO(str(last) if args.resume and last.exists() else args.weights)\n    resume=bool(args.resume and last.exists())\n    completed=resume and (model.ckpt.get(\'epoch\',0)<0 or model.ckpt.get(\'epoch\',0)+1>=args.epochs)\n    if completed and model.ckpt.get(\'epoch\',0)<0 and model.ckpt.get(\'train_args\',{}).get(\'epochs\')!=args.epochs:\n        raise ValueError(\'A completed checkpoint cannot resume a different epoch budget; use a new output experiment\')\n    if not completed:\n        if args.train_minutes:\n            # Ultralytics\' time= scheduler computes an epoch budget from zero.\n            # A callback limits added training while retaining the saved epoch/optimizer.\n            clock={}\n            def begin(trainer):clock[\'deadline\']=time.monotonic()+args.train_minutes*60\n            def limit(trainer):\n                if args.train_minutes and time.monotonic()>=clock[\'deadline\']:trainer.stop=True\n            model.add_callback(\'on_train_start\',begin)\n            model.add_callback(\'on_train_batch_end\',limit)\n        if args.train_minutes:\n            def preserve_resume(trainer):\n                if trainer.last.exists():\n                    resume_target=trainer.last.with_name(\'resume-last.pt\')\n                    if resume_target.exists():atomic_copy(resume_target,resume_target.with_name(\'resume-previous.pt\'))\n                    atomic_copy(trainer.last,resume_target)\n            model.add_callback(\'on_model_save\',preserve_resume)\n        model.train(data=str(data_file),epochs=args.epochs,\n                    imgsz=args.imgsz,batch=args.batch,device=args.device,seed=42,\n                    project=args.output,name=args.task,exist_ok=True,patience=8 if args.freeze else 30,\n                    close_mosaic=min(3 if args.freeze else 10,args.epochs),workers=2,amp=True,\n                    save_period=-1,resume=resume,freeze=args.freeze or None,\n                    cache=\'ram\' if args.cache_ram else False,\n                    time=None)\n    best=Path(args.output)/args.task/\'weights/best.pt\'\n    model=YOLO(str(best))\n    metrics=model.val(data=str(data_file),split=\'test\',\n                      imgsz=args.imgsz,batch=args.batch,device=args.device,\n                      project=str(output),name=args.task+\'-test\',exist_ok=True)\n    exported=Path(model.export(format=\'onnx\',imgsz=args.imgsz,batch=1,dynamic=False,\n                               simplify=False,opset=12,nms=False,device=\'cpu\'))\n    deployment={\'people\':\'person-yolo12m.onnx\',\'head_ppe\':\'ppe-head.onnx\',\'vest_ppe\':\'ppe-vest.onnx\'}[args.task]\n    import shutil\n    target=Path(args.output)/deployment;shutil.copyfile(exported,target)\n    if args.task==\'people\':shutil.copyfile(best,Path(args.output)/\'person-yolo12m.pt\')\n    result={\'task\':args.task,\'test_metrics\':{k:float(v) for k,v in metrics.results_dict.items()},\n            \'onnx_sha256\':hashlib.sha256(target.read_bytes()).hexdigest(),\n            \'linq_status\':\'not_compiled\',\'event_metrics\':\'not_evaluated\',\n            \'freeze_layers\':args.freeze,\'training_time_budget_minutes\':args.train_minutes,\n            \'continuation\':continuation,\n            \'actual_epochs\':sum(1 for _ in (output/args.task/\'results.csv\').open())-1}\n    (Path(args.output)/(args.task+\'-metrics.json\')).write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\n\n\ndef train_classifier(args):\n    import numpy as np\n    import torch\n    from torch.utils.data import DataLoader\n    from torchvision import datasets,models,transforms\n    import onnx\n    torch.manual_seed(42)\n    root=Path(args.data)/(args.task+\'_classifier\')\n    mean=[.485,.456,.406];std=[.229,.224,.225]\n    common=[transforms.Resize((224,224)),transforms.ToTensor(),transforms.Normalize(mean,std)]\n    train=datasets.ImageFolder(root/\'train\',transforms.Compose([\n        transforms.Resize((224,224)),transforms.RandomHorizontalFlip(),\n        transforms.ColorJitter(.15,.15,.15,.03),*common[1:]]),is_valid_file=valid_classifier_file)\n    val=datasets.ImageFolder(root/\'val\',transforms.Compose(common),is_valid_file=valid_classifier_file)\n    test=datasets.ImageFolder(root/\'test\',transforms.Compose(common),is_valid_file=valid_classifier_file)\n    if train.class_to_idx!=val.class_to_idx or train.class_to_idx!=test.class_to_idx:\n        raise ValueError(\'Every split must have the same labelled classes\')\n    if min(len(train),len(val),len(test))<20:raise ValueError(\'Not enough reviewed crops for independent splits\')\n    device=torch.device(\'cuda:\'+args.device if str(args.device).isdigit() else args.device)\n    if args.classifier==\'resnet152\':\n        model=models.resnet152(weights=models.ResNet152_Weights.DEFAULT)\n    else:model=models.resnet50(weights=models.ResNet50_Weights.DEFAULT)\n    model.fc=torch.nn.Linear(model.fc.in_features,len(train.classes));model.to(device)\n    if args.classifier_freeze_early:\n        for name,parameter in model.named_parameters():\n            parameter.requires_grad=name.startswith((\'layer4.\',\'fc.\'))\n    loaders={k:DataLoader(d,batch_size=args.batch,shuffle=k==\'train\',num_workers=2)\n             for k,d in [(\'train\',train),(\'val\',val),(\'test\',test)]}\n    weights=np.bincount(train.targets,minlength=len(train.classes))\n    loss_fn=torch.nn.CrossEntropyLoss(weight=torch.tensor(1/np.maximum(weights,1),dtype=torch.float32,device=device))\n    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=1e-4,weight_decay=1e-4)\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=device.type==\'cuda\')\n    folder=Path(args.output);folder.mkdir(parents=True,exist_ok=True)\n    best_loss=float(\'inf\');checkpoint=folder/(args.task+\'-classifier.pt\')\n    start_epoch=0;last=folder/(args.task+\'-classifier-last.pt\')\n    if args.resume and last.exists():\n        state=torch.load(last,map_location=device,weights_only=True)\n        if state[\'classes\']!=train.classes:raise ValueError(\'Resume class order differs\')\n        if state.get(\'architecture\',args.classifier)!=args.classifier:raise ValueError(\'Resume classifier architecture differs\')\n        if state.get(\'freeze_early\',args.classifier_freeze_early)!=args.classifier_freeze_early:raise ValueError(\'Resume optimizer parameter groups differ\')\n        model.load_state_dict(state[\'state_dict\']);optimizer.load_state_dict(state[\'optimizer\'])\n        if state.get(\'scaler\'):scaler.load_state_dict(state[\'scaler\'])\n        if state.get(\'torch_rng\') is not None:torch.set_rng_state(state[\'torch_rng\'].cpu())\n        if state.get(\'cuda_rng\') and device.type==\'cuda\':torch.cuda.set_rng_state_all([v.cpu() for v in state[\'cuda_rng\']])\n        start_epoch=state[\'epoch\'];best_loss=state[\'best_loss\']\n    def evaluate(loader):\n        model.eval();losses=[];scores=[];targets=[]\n        with torch.no_grad():\n            for x,y in loader:\n                x,y=x.to(device),y.to(device);logits=model(x)\n                losses.append(float(loss_fn(logits,y)));scores.extend(logits.softmax(1).cpu().numpy());targets.extend(y.cpu().numpy())\n        return float(np.mean(losses)),np.asarray(scores),np.asarray(targets)\n    deadline=time.monotonic()+args.train_minutes*60 if args.train_minutes else float(\'inf\')\n    actual_epochs=start_epoch\n    for epoch in range(start_epoch,args.epochs):\n        model.train()\n        if args.classifier_freeze_early:\n            for name,module in model.named_children():\n                if name not in {\'layer4\',\'fc\'}:module.eval()\n        for x,y in loaders[\'train\']:\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type,enabled=device.type==\'cuda\'):\n                loss=loss_fn(model(x.to(device)),y.to(device))\n            scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()\n            if time.monotonic()>=deadline:break\n        validation_loss,_,_=evaluate(loaders[\'val\'])\n        print(json.dumps({\'epoch\':epoch+1,\'val_loss\':validation_loss}),flush=True)\n        if validation_loss<best_loss:\n            best_loss=validation_loss;atomic_torch_save({\'state_dict\':model.state_dict(),\'classes\':train.classes},checkpoint)\n        atomic_torch_save({\'state_dict\':model.state_dict(),\'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict(),\n                    \'epoch\':epoch+1,\'best_loss\':best_loss,\'classes\':train.classes,\n                    \'architecture\':args.classifier,\'freeze_early\':args.classifier_freeze_early,\n                    \'torch_rng\':torch.get_rng_state(),\n                    \'cuda_rng\':torch.cuda.get_rng_state_all() if device.type==\'cuda\' else []},last,previous=True)\n        actual_epochs=epoch+1\n        if time.monotonic()>=deadline:\n            print(\'Training time budget reached; selecting validated checkpoint and exporting\',flush=True)\n            break\n    saved=torch.load(checkpoint,map_location=device,weights_only=True);model.load_state_dict(saved[\'state_dict\'])\n    _,probabilities,truth=evaluate(loaders[\'val\'])\n    choices=[]\n    for threshold in np.linspace(.5,.99,50):\n        selected=probabilities.max(1)>=threshold;pred=probabilities.argmax(1)\n        per_class=[]\n        for label in range(len(train.classes)):\n            positives=selected & (pred==label);count=int(positives.sum())\n            per_class.append((count,int(((truth==label)&positives).sum())/max(count,1)))\n        if all(count>=20 and precision>=.95 for count,precision in per_class):\n            choices.append((float(selected.mean()),float(threshold)))\n    threshold=max(choices)[1] if choices else .8\n    _,test_scores,test_truth=evaluate(loaders[\'test\']);test_pred=test_scores.argmax(1)\n    # Rejections count as misses for recall on assessable test crops.\n    selected=test_scores.max(1)>=threshold;stats={}\n    for label,name in enumerate(train.classes):\n        tp=int(((test_pred==label)&(test_truth==label)&selected).sum())\n        fp=int(((test_pred==label)&(test_truth!=label)&selected).sum())\n        fn=int(((test_truth==label)&((test_pred!=label)|~selected)).sum())\n        precision=tp/max(tp+fp,1);recall=tp/max(tp+fn,1)\n        stats[name]={\'precision\':precision,\'recall\':recall,\'f1\':2*precision*recall/max(precision+recall,1e-12),\'tp\':tp,\'fp\':fp,\'fn\':fn}\n    name=\'ppe-head-classifier.onnx\' if args.task==\'head_ppe\' else \'ppe-torso-classifier.onnx\'\n    target=folder/name;model.eval().cpu()\n    torch.onnx.export(model,torch.zeros(1,3,224,224),str(target),input_names=[\'images\'],\n                      output_names=[\'logits\'],opset_version=12,dynamo=False)\n    graph=onnx.load(target)\n    for key,value in {\'class_names\':json.dumps(train.classes),\'rgb_mean\':json.dumps(mean),\'rgb_std\':json.dumps(std),\n                      \'abstain_threshold\':str(threshold),\'crop_policy\':\'proposal_context_1.3\',\n                      \'resize_algorithm\':\'pil_bilinear\'}.items():\n        item=graph.metadata_props.add();item.key=key;item.value=value\n    onnx.save(graph,target)\n    report={\'classes\':train.classes,\'threshold\':threshold,\'validation_precision_target_met\':bool(choices),\n            \'actual_epochs\':actual_epochs,\'training_time_budget_minutes\':args.train_minutes,\n            \'frozen_early_layers\':args.classifier_freeze_early,\n            \'test\':stats,\'test_coverage\':float(selected.mean()),\'linq_status\':\'not_compiled\',\n            \'qc_excluded_image_sha256\':sorted(QC_EXCLUDED_IMAGE_SHA256),\n            \'split_crops\':{\'train\':len(train),\'val\':len(val),\'test\':len(test)},\n            \'warning\':\'Crop classification metrics are not event metrics; unobservable crops need a separate evaluation set.\'}\n    (folder/(args.task+\'-classifier-metrics.json\')).write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',default=\'data/datasets/ppe-prepared\')\n    p.add_argument(\'--task\',choices=[\'people\',\'head_ppe\',\'vest_ppe\'],required=True)\n    p.add_argument(\'--mode\',choices=[\'detect\',\'classify\'],default=\'detect\')\n    p.add_argument(\'--weights\',default=\'yolo12m.pt\');p.add_argument(\'--classifier\',choices=[\'resnet50\',\'resnet152\'],default=\'resnet50\')\n    p.add_argument(\'--epochs\',type=int,default=150);p.add_argument(\'--imgsz\',type=int,default=640)\n    p.add_argument(\'--batch\',type=int,default=4);p.add_argument(\'--device\',default=\'0\')\n    p.add_argument(\'--resume\',action=\'store_true\')\n    p.add_argument(\'--freeze\',type=int,default=0)\n    p.add_argument(\'--cache-ram\',action=\'store_true\')\n    p.add_argument(\'--classifier-freeze-early\',action=\'store_true\')\n    p.add_argument(\'--train-minutes\',type=float,default=0)\n    p.add_argument(\'--continue-checkpoint\',help=\'Continue our saved detector optimizer/epoch state with an added time budget\')\n    p.add_argument(\'--output\',default=\'training-output\');a=p.parse_args()\n    if a.mode==\'classify\' and a.task==\'people\':p.error(\'Classifier is only for head/torso PPE\')\n    (train_detector if a.mode==\'detect\' else train_classifier)(a)\n',encoding='utf-8')
(SCRIPTS/'run_ppe_training.py').write_text('"""Run independent tasks on up to two GPUs, keeping logs and resume state."""\nimport argparse\nfrom concurrent.futures import ThreadPoolExecutor,wait\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\nimport threading\nimport gc\nimport time\n\n\ndef run(args):\n    bounded=args.fast or args.night\n    deadline=time.monotonic()+args.max_minutes*60 if bounded else float(\'inf\')\n    import torch\n    if not torch.cuda.is_available():raise RuntimeError(\'Training requires a CUDA GPU\')\n    devices=list(range(min(torch.cuda.device_count(),2)))\n    queues=[[(\'detect\',\'people\'),(\'detect\',\'vest_ppe\'),(\'classify\',\'head_ppe\')],\n            [(\'detect\',\'head_ppe\'),(\'classify\',\'vest_ppe\')]]\n    if args.fast:queues[0]=[(\'detect\',\'vest_ppe\'),(\'detect\',\'people\'),(\'classify\',\'head_ppe\')]\n    if args.night:queues[0]=[(\'detect\',\'people\'),(\'detect\',\'vest_ppe\'),(\'classify\',\'head_ppe\')]\n    if len(devices)==1:queues=[queues[0]+queues[1]]\n    if args.jobs:\n        requested=set(args.jobs)\n        queues=[[job for job in queue if \'-\'.join(job) in requested] for queue in queues]\n    checkpoint_weights={};lineage={}\n    if bounded and not args.checkpoint_root:\n        raise ValueError(\'Fast continuation requires --checkpoint-root; refusing to restart trained tasks from stock weights\')\n    if args.checkpoint_root:\n        from ultralytics import YOLO\n        for task,nc in [(\'people\',1),(\'head_ppe\',2)]:\n            checkpoint=Path(args.checkpoint_root)/task/\'weights/last.pt\'\n            if not checkpoint.is_file():raise FileNotFoundError(f\'Required saved checkpoint is missing: {checkpoint}\')\n            loaded=YOLO(str(checkpoint));state=loaded.ckpt\n            if state.get(\'epoch\',-1)<0 or not state.get(\'optimizer\'):\n                raise ValueError(f\'{task}: expected a partial checkpoint with optimizer state\')\n            if len(loaded.names)!=nc:raise ValueError(f\'{task}: checkpoint classes differ\')\n            lineage[task]={\'path\':str(checkpoint),\'sha256\':hashlib.sha256(checkpoint.read_bytes()).hexdigest(),\n                           \'completed_epochs\':int(state[\'epoch\'])+1,\n                           \'mode\':\'resume optimizer and epoch state; limit additional training time\'}\n            checkpoint_weights[task]=str(checkpoint)\n            print(\'RESTORED\',task,\'completed epochs\',lineage[task][\'completed_epochs\'],flush=True)\n            del loaded,state;gc.collect()\n    weights=None\n    if any(mode==\'detect\' for queue in queues for mode,_ in queue):\n        from ultralytics import YOLO\n        print(\'Preparing YOLO12m initialization once before parallel workers\',flush=True)\n        initial=YOLO(\'yolo12m.pt\');weights=str(Path(initial.ckpt_path).resolve())\n        del initial;gc.collect()\n        # Ultralytics\' AMP sanity check loads this tiny official checkpoint.\n        # Fetch it once so concurrent trainers cannot overwrite one download.\n        initial=YOLO(\'yolo26n.pt\');del initial;gc.collect()\n    if any(mode==\'classify\' for queue in queues for mode,_ in queue):\n        from torchvision import models\n        print(\'Preparing classifier initialization once before parallel workers\',flush=True)\n        enum=models.ResNet152_Weights if args.classifier==\'resnet152\' else models.ResNet50_Weights\n        initial=enum.DEFAULT.get_state_dict(progress=True,check_hash=True)\n        del initial;gc.collect()\n    output=Path(args.output);output.mkdir(parents=True,exist_ok=True)\n    if lineage:(output/\'checkpoint-lineage.json\').write_text(json.dumps(lineage,indent=2)+\'\\n\')\n    script=Path(__file__).with_name(\'train_ppe.py\')\n    manifest=Path(args.data)/\'manifest.jsonl\'\n    identity={\'data\':str(Path(args.data).resolve()),\'epochs\':args.epochs,\'classifier_epochs\':args.classifier_epochs,\n              \'classifier\':args.classifier,\'batch\':args.batch,\'classifier_batch\':args.classifier_batch,\'imgsz\':args.imgsz,\n              \'fast\':args.fast,\'night\':args.night,\'job_minutes\':args.job_minutes,\'max_minutes\':args.max_minutes,\n              \'checkpoint_lineage\':lineage,\n              \'trainer_sha256\':hashlib.sha256(script.read_bytes()).hexdigest(),\n              \'data_manifest_sha256\':hashlib.sha256(manifest.read_bytes()).hexdigest() if manifest.exists() else \'bounded-smoke-subset\'}\n    progress=output/\'training-progress.json\';lock=threading.Lock()\n    prior=json.loads(progress.read_text()) if progress.exists() else {}\n    state={\'identity\':identity,\'jobs\':prior.get(\'jobs\',{}) if prior.get(\'identity\')==identity else {},\n           \'gpus\':[torch.cuda.get_device_name(i) for i in devices]}\n    def save():\n        temp=progress.with_suffix(\'.tmp\');temp.write_text(json.dumps(state,indent=2)+\'\\n\');temp.replace(progress)\n    def queue(device,jobs):\n        for mode,task in jobs:\n            key=mode+\'-\'+task\n            if state[\'jobs\'].get(key,{}).get(\'status\')==\'done\':\n                print(\'SKIP completed\',key,flush=True);continue\n            if deadline-time.monotonic()<120:\n                with lock:\n                    state[\'jobs\'][key]={\'status\':\'skipped_deadline\',\'gpu\':device};save()\n                print(\'SKIPPED deadline\',key,flush=True);continue\n            log=output/(key+\'.log\')\n            budget=({\'detect-people\':20,\'detect-head_ppe\':175,\'detect-vest_ppe\':105,\n                     \'classify-head_ppe\':55,\'classify-vest_ppe\':40}.get(key,args.job_minutes) if args.night else args.job_minutes)\n            budget=min(budget,max(1,(deadline-time.monotonic())/60-15)) if bounded else budget\n            epochs=25 if args.night and task==\'people\' else args.epochs\n            cmd=[sys.executable,str(script),\'--data\',args.data,\'--output\',args.output,\'--task\',task,\n                 \'--mode\',mode,\'--device\',str(device),\'--epochs\',str(epochs if mode==\'detect\' else args.classifier_epochs),\n                 \'--batch\',str(args.batch if mode==\'detect\' else args.classifier_batch),\'--imgsz\',str(args.imgsz),\'--resume\']\n            if mode==\'classify\':cmd+=[\'--classifier\',args.classifier]\n            elif weights:cmd+=[\'--weights\',checkpoint_weights.get(task,weights)]\n            if bounded:\n                cmd+=[\'--train-minutes\',str(budget)]\n                if mode==\'classify\':cmd+=[\'--classifier-freeze-early\']\n                elif task in checkpoint_weights:cmd+=[\'--continue-checkpoint\',checkpoint_weights[task],\'--cache-ram\']\n                else:cmd+=[\'--cache-ram\']+([\'--freeze\',\'10\'] if args.fast else [])\n            with lock:\n                state[\'jobs\'][key]={\'status\':\'running\',\'gpu\':device,\'log\':log.name,\'training_budget_minutes\':budget};save()\n            print(\'START\',key,\'GPU\',device,flush=True)\n            env={**os.environ,\'PYTHONUNBUFFERED\':\'1\',\'OMP_NUM_THREADS\':\'2\',\'MKL_NUM_THREADS\':\'2\'}\n            with log.open(\'a\',encoding=\'utf-8\') as f:\n                try:\n                    result=subprocess.run(cmd,stdout=f,stderr=subprocess.STDOUT,env=env,\n                        timeout=min(deadline-time.monotonic(),(budget+25)*60) if bounded else None)\n                    returncode=result.returncode\n                except subprocess.TimeoutExpired:\n                    returncode=124\n                    print(\'Worker exceeded deadline; saved checkpoints are retained\',file=f,flush=True)\n            with lock:\n                state[\'jobs\'][key].update(status=\'done\' if returncode==0 else \'failed\',returncode=returncode);save()\n            print(\'DONE\' if returncode==0 else \'FAILED\',key,\'GPU\',device,flush=True)\n    with lock:save()\n    with ThreadPoolExecutor(max_workers=len(devices)) as pool:\n        pending={pool.submit(queue,device,jobs) for device,jobs in zip(devices,queues)}\n        while pending:\n            done,pending=wait(pending,timeout=20)\n            for future in done:future.result()\n            with lock:running=[(key,dict(job)) for key,job in state[\'jobs\'].items() if job[\'status\']==\'running\']\n            for key,job in running:\n                try:\n                    with (output/job[\'log\']).open(\'rb\') as f:\n                        f.seek(0,2);f.seek(max(0,f.tell()-3000));tail=f.read().decode(\'utf-8\',errors=\'replace\')\n                    lines=re.sub(r\'\\x1b\\[[0-9;]*[A-Za-z]\',\'\',tail).replace(\'\\r\',\'\\n\').splitlines()\n                    print(key,\'GPU\',job[\'gpu\'],next((s.strip()[:220] for s in reversed(lines) if s.strip()),\'initializing\'),flush=True)\n                except OSError:pass\n    if any(job[\'status\']!=\'done\' for job in state[\'jobs\'].values()):\n        raise RuntimeError(\'One or more tasks failed; see training-progress.json and per-task logs\')\n    print(\'All requested tasks completed. Independent event quality remains unmeasured.\',flush=True)\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--data\',required=True);p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--epochs\',type=int,default=150);p.add_argument(\'--classifier-epochs\',type=int,default=30)\n    p.add_argument(\'--imgsz\',type=int,default=640);p.add_argument(\'--batch\',type=int,default=4)\n    p.add_argument(\'--classifier-batch\',type=int,default=16)\n    p.add_argument(\'--classifier\',choices=[\'resnet50\',\'resnet152\'],default=\'resnet152\')\n    p.add_argument(\'--fast\',action=\'store_true\',help=\'Freeze pretrained early layers and limit each training stage\')\n    p.add_argument(\'--night\',action=\'store_true\',help=\'Five tasks: short person continuation, PPE detectors, ResNet50 classifiers\')\n    p.add_argument(\'--job-minutes\',type=float,default=15)\n    p.add_argument(\'--max-minutes\',type=float,default=75,help=\'Fast-mode overall worker deadline, including setup/export\')\n    p.add_argument(\'--checkpoint-root\',help=\'Saved training-output folder; required for fast continuation\')\n    p.add_argument(\'--jobs\',nargs=\'+\',choices=[\'detect-people\',\'detect-head_ppe\',\'detect-vest_ppe\',\'classify-head_ppe\',\'classify-vest_ppe\'])\n    run(p.parse_args())\n',encoding='utf-8')
(SCRIPTS/'training_checkpoint.py').write_text('"""Crash-safe replacement of our own training checkpoints on one filesystem."""\nimport os\nfrom pathlib import Path\nimport shutil\n\n\ndef atomic_copy(source, target):\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = target.with_name(target.name + \'.writing\')\n    with Path(source).open(\'rb\') as incoming, temporary.open(\'wb\') as outgoing:\n        shutil.copyfileobj(incoming, outgoing)\n        outgoing.flush()\n        os.fsync(outgoing.fileno())\n    os.replace(temporary, target)\n\n\ndef atomic_torch_save(state, target, previous=False):\n    import torch\n    target = Path(target)\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = target.with_name(target.name + \'.writing\')\n    with temporary.open(\'wb\') as stream:\n        torch.save(state, stream)\n        stream.flush()\n        os.fsync(stream.fileno())\n    if previous and target.exists():\n        atomic_copy(target, target.with_name(target.stem + \'-previous.pt\'))\n    os.replace(temporary, target)\n',encoding='utf-8')
(SCRIPTS/'check_linq_compatibility.py').write_text('"""Check the documented YOLO12m split and optionally compile it with LinQ SDK.\n\nNo compatibility claim is made from ONNX checks alone. Run --compile only in\nthe separate Linux/Python 3.12 SDK environment. Calibration is train-only NPZ.\n"""\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport sys\nfrom pathlib import Path\n\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\n\nimport numpy as np\nimport onnx\nimport onnxruntime as ort\n\nCUTS = [\'/model.21/Div_1_output_0\', \'/model.21/Sub_1_output_0\',\n        \'/model.21/Sigmoid_output_0\']\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef preflight(model, output, images=()):\n    output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    graph = onnx.load(str(model)); onnx.checker.check_model(graph)\n    inputs = [v.name for v in graph.graph.input]\n    outputs = [v.name for v in graph.graph.output]\n    available = {v for n in graph.graph.node for v in n.output}\n    if not set(CUTS) <= available:\n        raise ValueError(\'This export differs from the documented YOLO12m graph: missing cut tensors\')\n    dims = [[d.dim_value for d in v.type.tensor_type.shape.dim] for v in graph.graph.input]\n    if len(inputs) != 1 or any(d <= 0 for shape in dims for d in shape):\n        raise ValueError(\'One static input is required\')\n    head, tail = output/\'yolo12m-head.onnx\', output/\'yolo12m-tail.onnx\'\n    onnx.utils.extract_model(str(model), str(head), inputs, CUTS)\n    onnx.utils.extract_model(str(model), str(tail), CUTS, outputs)\n    options = ort.SessionOptions(); options.intra_op_num_threads = 4\n    sessions = [ort.InferenceSession(str(p), options, providers=[\'CPUExecutionProvider\'])\n                for p in (model, head, tail)]\n    # Random data checks graph wiring only; it is NEVER calibration data.\n    tensors = [np.random.default_rng(42).random(dims[0], dtype=np.float32)]\n    if images:\n        import cv2\n        from app.scenario_models import letterbox_tensor\n        for path in images:\n            image = cv2.imread(str(path))\n            if image is None: raise ValueError(f\'Cannot read {path}\')\n            tensors.append(letterbox_tensor(image, (dims[0][3], dims[0][2]))[0])\n    parity = []\n    for tensor in tensors:\n        full = sessions[0].run(None, {inputs[0]: tensor})\n        intermediate = sessions[1].run(None, {inputs[0]: tensor})\n        restored = sessions[2].run(None, dict(zip(CUTS, intermediate)))\n        for actual, expected in zip(restored, full):\n            np.testing.assert_allclose(actual, expected, rtol=1e-5, atol=1e-4)\n        parity.append(max(float(np.max(np.abs(a-b))) for a,b in zip(full, restored)))\n    report = {\'model_sha256\': sha(model), \'onnx_ir\': graph.ir_version,\n              \'opsets\': {x.domain: x.version for x in graph.opset_import},\n              \'input_names\': inputs, \'input_shapes\': dims, \'cut_tensors\': CUTS,\n              \'head_sha256\': sha(head), \'tail_sha256\': sha(tail),\n              \'parity_max_abs_errors\': parity, \'parity_samples\': len(tensors),\n              \'real_image_samples\': len(tensors)-1, \'status\': \'onnx_split_verified\',\n              \'sdk_compiled\': False, \'hardware_verified\': False,\n              \'warning\': \'ONNX split parity is not SDK compilation, hardware parity or accuracy evaluation.\'}\n    return report\n\n\ndef compile_head(output, calibration):\n    if platform.system() != \'Linux\' or sys.version_info[:2] != (3,12):\n        raise RuntimeError(\'LinQ binary SDK needs Linux x86_64 / Python 3.12\')\n    from tpu_framework import RegularModel, Network, compiler, TPU_H1M, O1, TpuProgram\n    from tpu_framework import preliminary_model_conditioning\n    from dnn_quant.common import SupportedLayerTypes as Slt\n    from dnn_quant.autoquant.cfg_wrappers.cfg_quant_block_wrappers import FLOAT16, PIECEWISE_LUT\n    data = np.load(calibration, allow_pickle=False)\n    if \'images\' not in data or \'split\' not in data or str(data[\'split\'].item()) != \'train\':\n        raise ValueError(\'Calibration NPZ must contain images and split="train"\')\n    images = data[\'images\'].astype(np.float32)\n    graph = preliminary_model_conditioning(onnx.load(str(output/\'yolo12m-head.onnx\')))\n    shape = [d.dim_value for d in graph.graph.input[0].type.tensor_type.shape.dim]\n    if images.ndim != 4 or list(images.shape[1:]) != shape[1:] or len(images) < 8:\n        raise ValueError(\'Need >=8 preprocessed train samples matching the static CHW input\')\n    if not np.isfinite(images).all() or images.min()<0 or images.max()>1:\n        raise ValueError(\'Calibration requires finite RGB /255 letterboxed tensors\')\n    regular = RegularModel(model=graph, operation_cfg={Slt.FW_LAYER_REDUCE_MAX: FLOAT16,\n                           Slt.FW_LAYER_REDUCE_MEAN: FLOAT16, Slt.FW_LAYER_REDUCE_SUM: FLOAT16},\n                           swish_layer_mode=PIECEWISE_LUT)\n    name = graph.graph.input[0].name\n    name = regular.anchors_mapping.get(name, name)\n    thresholds = regular.calibrate_torch(calibration_data={name: images}, device=\'cpu\')\n    regular.dump_thresholds(str(output/\'thresholds.json\'), thresholds)\n    quantized = regular.quantize(thresholds)\n    quantized.save(str(output), \'yolo12m-head.qm.onnx\')\n    network, _ = Network.from_quantized_model(quantized)\n    executable, descriptions = compiler.compile_(hardware_parameters=TPU_H1M, network=network, parameters=O1)\n    target = output/\'yolo12m-head-b1-o1-h1m.tpu\'\n    TpuProgram.from_executable(executable, descriptions).to_file(str(target))\n    return {\'sdk_compiled\': True, \'status\': \'compiled_unverified\', \'program_sha256\': sha(target),\n            \'calibration_samples\': len(images), \'calibration_sha256\': sha(calibration),\n            \'hardware_verified\': False}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--model\', default=\'data/models/yolo12m.onnx\')\n    parser.add_argument(\'--output\', default=\'data/linq/yolo12m\')\n    parser.add_argument(\'--report\', default=\'docs/yolo12m-compatibility.json\')\n    parser.add_argument(\'--image\', action=\'append\', default=[])\n    parser.add_argument(\'--compile\', action=\'store_true\')\n    parser.add_argument(\'--calibration\')\n    args = parser.parse_args()\n    report = preflight(args.model, args.output, args.image)\n    code = 0\n    if args.compile:\n        try:\n            if not args.calibration: raise ValueError(\'--calibration is required\')\n            report.update(compile_head(Path(args.output), args.calibration))\n        except Exception as exc:\n            report.update(status=\'compile_failed\', compile_error=f\'{type(exc).__name__}: {exc}\')\n            code = 1\n    Path(args.report).parent.mkdir(parents=True, exist_ok=True)\n    Path(args.report).write_text(json.dumps(report, indent=2)+\'\\n\', encoding=\'utf-8\')\n    print(json.dumps(report, indent=2)); return code\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n',encoding='utf-8')
(SCRIPTS/'prepare_fast_ppe.py').write_text('"""Create a bounded, reproducible experiment without changing the full corpus.\n\nSelection uses source, labels and image identifiers, never model predictions.\nNative train/val/test membership is retained. This is an exploratory subset.\n"""\nimport argparse\nfrom collections import defaultdict, deque\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport time\n\nfrom train_ppe import QC_EXCLUDED_IMAGE_SHA256\n\n\ndef rank(path):\n    return hashlib.sha256((\'linq-fast-v1/42/\'+path.name).encode()).digest()\n\n\ndef choose(paths, limit, metadata, task, crop=False):\n    buckets=defaultdict(list)\n    supported={\'people\':{\'person\'},\'head_ppe\':{\'hardhat\',\'no_hardhat\'},\n               \'vest_ppe\':{\'vest\',\'no_vest\'}}[task]\n    for path in paths:\n        digest=path.stem[:64]\n        if crop and digest in QC_EXCLUDED_IMAGE_SHA256:continue\n        row=metadata[digest]\n        labels=tuple(sorted({x[0] for x in row[\'labels\']} & supported))\n        areas=[x[3]*x[4] for x in row[\'labels\'] if x[0] in supported]\n        size=\'small\' if areas and min(areas)<.01 else \'other\'\n        buckets[(row[\'source\'],labels,size)].append(path)\n    buckets=[deque(sorted(values,key=rank)) for _,values in sorted(buckets.items())]\n    selected=[];per_image=defaultdict(int)\n    while buckets and len(selected)<limit:\n        remaining=[]\n        for bucket in buckets:\n            while bucket:\n                path=bucket.popleft();digest=path.stem[:64]\n                if not crop or per_image[digest]<4:\n                    selected.append(path);per_image[digest]+=1;break\n            if bucket:remaining.append(bucket)\n            if len(selected)>=limit:break\n        buckets=remaining\n    return selected\n\n\ndef prepare(source, output, workers=8, tiny=False):\n    if output.exists():raise ValueError(\'Use a new output directory; do not mix experiment subsets\')\n    output.mkdir(parents=True)\n    metadata={r[\'sha256\']:r for r in map(json.loads,(source/\'manifest.jsonl\').read_text().splitlines())}\n    jobs=[];selected=[];counts={};groups={};started=time.monotonic()\n    train_limits={\'people\':1200,\'head_ppe\':2400,\'vest_ppe\':2020}\n    for task in train_limits:\n        dest=output/task;dest.mkdir()\n        shutil.copyfile(source/task/\'data.yaml\',dest/\'data.yaml\')\n        for split in [\'train\',\'val\',\'test\']:\n            limit=(train_limits[task] if split==\'train\' else 400) if not tiny else 24\n            files=choose(list((source/task/\'images\'/split).glob(\'*\')),limit,metadata,task)\n            counts[f\'{task}/{split}\']=len(files)\n            for image in files:\n                digest=image.stem[:64];row=metadata[digest]\n                if row[\'split\']!=split:raise ValueError(\'Source split mismatch\')\n                group=(row[\'source\'],row[\'group\'])\n                if group in groups and groups[group]!=split:raise ValueError(\'Source-group leakage\')\n                groups[group]=split\n                label=source/task/\'labels\'/split/(image.stem+\'.txt\')\n                jobs.extend([(image,dest/\'images\'/split/image.name),(label,dest/\'labels\'/split/label.name)])\n                selected.append({\'task\':task,\'split\':split,\'file\':image.name,\'sha256\':digest})\n    for task,limit in [(\'head_ppe\',2000),(\'vest_ppe\',1600)]:\n        for split in [\'train\',\'val\',\'test\']:\n            for folder in sorted((source/(task+\'_classifier\')/split).iterdir()):\n                max_count=(limit if split==\'train\' else 256) if not tiny else 12\n                files=choose(list(folder.glob(\'*.jpg\')),max_count,metadata,task,crop=True)\n                counts[f\'{task}_classifier/{split}/{folder.name}\']=len(files)\n                for image in files:\n                    digest=image.stem[:64];row=metadata[digest]\n                    if row[\'split\']!=split:raise ValueError(\'Crop split mismatch\')\n                    group=(row[\'source\'],row[\'group\'])\n                    if group in groups and groups[group]!=split:raise ValueError(\'Crop source-group leakage\')\n                    groups[group]=split\n                    target=output/(task+\'_classifier\')/split/folder.name/image.name\n                    jobs.append((image,target))\n                    selected.append({\'task\':task+\'_classifier\',\'split\':split,\'class\':folder.name,\n                                     \'file\':image.name,\'sha256\':digest})\n    for _,target in jobs:target.parent.mkdir(parents=True,exist_ok=True)\n    def copy(pair):\n        src,dst=pair;shutil.copyfile(src,dst)\n        if dst.stat().st_size!=src.stat().st_size:raise IOError(f\'Copy size mismatch: {src.name}\')\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        futures=[pool.submit(copy,pair) for pair in jobs]\n        for index,future in enumerate(as_completed(futures),1):\n            future.result()\n            if index%500==0:print(f\'Local data cache {index}/{len(jobs)} files\',flush=True)\n    digests={x[\'sha256\'] for x in selected}\n    (output/\'manifest.jsonl\').write_text(\'\\n\'.join(json.dumps(metadata[x]) for x in sorted(digests))+\'\\n\')\n    (output/\'selection.jsonl\').write_text(\'\\n\'.join(map(json.dumps,selected))+\'\\n\')\n    report={\'policy\':\'fast-v1-seed42-source-label-size-round-robin\',\'counts\':counts,\n            \'source_manifest_sha256\':hashlib.sha256((source/\'manifest.jsonl\').read_bytes()).hexdigest(),\n            \'selection_sha256\':hashlib.sha256((output/\'selection.jsonl\').read_bytes()).hexdigest(),\n            \'source_group_disjoint\':True,\'heldout_membership_preserved\':True,\n            \'test_scope\':\'fixed bounded subset, not the entire held-out corpus\',\n            \'max_classifier_crops_per_source_image_per_class\':4,\n            \'elapsed_seconds\':time.monotonic()-started,\n            \'warning\':\'Short fine-tuning experiment; no >0.95 quality guarantee.\'}\n    (output/\'fast-data-summary.json\').write_text(json.dumps(report,indent=2)+\'\\n\')\n    print(json.dumps(report,indent=2),flush=True)\n    return report\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--source\',required=True);p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--workers\',type=int,default=8);p.add_argument(\'--tiny\',action=\'store_true\')\n    a=p.parse_args();prepare(Path(a.source),Path(a.output),a.workers,a.tiny)\n',encoding='utf-8')

In [ ]:
import time
MODE='fast'
DEADLINE=time.monotonic()+110*60
IMGSZ=640
BATCH=16
EPOCHS=40  # Maximum total, including restored epochs; time callback may stop sooner.
CLASSIFIER_EPOCHS=4
LABELS_REVIEWED=True  # 96 sampled train crops reviewed, not the complete corpus.
OUTPUT=WORK/'fast-training-output';OUTPUT.mkdir(exist_ok=True)

In [ ]:
input_root=Path('/kaggle/input')
manifests=sorted(input_root.rglob('manifest.jsonl'))
assert manifests, 'Attach the prepared expanded Dataset; no slow download fallback in fast mode.'
FULL_DATA=manifests[0].parent
checkpoint_roots=[p.parent for p in input_root.rglob('training-progress.json')
                  if (p.parent/'people/weights/last.pt').is_file() and (p.parent/'head_ppe/weights/last.pt').is_file()]
assert len(checkpoint_roots)==1, 'Attach output of saved version 361611151 with both last.pt checkpoints. No fresh restart.'
CHECKPOINT_ROOT=checkpoint_roots[0]
print('Saved checkpoint folder:',CHECKPOINT_ROOT,flush=True)
DATA=WORK/'fast-data'
subprocess.run([sys.executable,str(SCRIPTS/'prepare_fast_ppe.py'),'--source',str(FULL_DATA),
                '--output',str(DATA),'--workers','8'],check=True,timeout=25*60)
shutil.copyfile(DATA/'fast-data-summary.json',OUTPUT/'fast-data-summary.json')
shutil.copyfile(DATA/'selection.jsonl',OUTPUT/'selection.jsonl')
print('Fast local dataset:',DATA,flush=True)

Перед полным запуском просмотрите случайные crop и проверьте классы. Особенно важны
`no_hardhat` и `no_vest`. Слишком маленькая или закрытая область должна попадать
в отдельный вручную размеченный набор видимости. Notebook не создаёт фиктивный
класс unknown. До появления такого набора используется отказ по порогу уверенности.
Автоматически исключаются crop меньше 24 px по стороне / 1024 px по площади,
крайние пропорции и сильное размытие. Это не проверяет правильность исходной разметки.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
rng=np.random.default_rng(42)
paths=list((DATA/'head_ppe_classifier/train').rglob('*.jpg'))+list((DATA/'vest_ppe_classifier/train').rglob('*.jpg'))
sample=rng.choice(len(paths),min(16,len(paths)),replace=False)
fig,axes=plt.subplots(4,4,figsize=(12,12))
for ax,index in zip(axes.flat,sample):
    p=paths[int(index)];ax.imshow(Image.open(p));ax.set_title(p.parent.name);ax.axis('off')
plt.tight_layout()

In [ ]:
print('Fast continuation: saved optimizer/epochs for people and head, 15 added minutes per task',flush=True)

In [ ]:
remaining=max(1,min(75,(DEADLINE-time.monotonic())/60-10))
training=subprocess.run([sys.executable,str(SCRIPTS/'run_ppe_training.py'),'--data',str(DATA),
                '--epochs',str(EPOCHS),'--classifier-epochs',str(CLASSIFIER_EPOCHS),
                '--imgsz',str(IMGSZ),'--batch',str(BATCH),'--classifier','resnet50',
                '--classifier-batch','32','--fast','--job-minutes','15','--max-minutes',str(remaining),
                '--checkpoint-root',str(CHECKPOINT_ROOT),
                '--output',str(OUTPUT)])
print('Training exit code:',training.returncode,'Check training-progress.json for all five tasks.',flush=True)

In [ ]:
# Train-only representative calibration tensors. No test images enter this file.
import cv2
for task in ['people','head_ppe','vest_ppe']:
    files=sorted((DATA/task/'images/train').glob('*'))
    rng=np.random.default_rng(42);chosen=rng.choice(len(files),min(64,len(files)),replace=False)
    tensor=[]
    for index in chosen:
        image=cv2.imread(str(files[int(index)]));h,w=image.shape[:2]
        scale=min(IMGSZ/w,IMGSZ/h);nw,nh=round(w*scale),round(h*scale)
        left,top=round((IMGSZ-nw)/2-.1),round((IMGSZ-nh)/2-.1)
        padded=np.full((IMGSZ,IMGSZ,3),114,np.uint8)
        padded[top:top+nh,left:left+nw]=cv2.resize(image,(nw,nh))
        tensor.append(padded[:,:,::-1].transpose(2,0,1).astype(np.float32)/255)
    np.savez_compressed(OUTPUT/f'calibration-{task}-train.npz',images=np.stack(tensor),split=np.array('train'),
                        source_image_sha256=np.array([files[int(i)].stem for i in chosen]))
    del tensor
# Split parity is local ONNX verification; SDK compilation is performed separately.
if (OUTPUT/'person-yolo12m.onnx').exists():
    subprocess.run([sys.executable,str(SCRIPTS/'check_linq_compatibility.py'),
                '--model',str(OUTPUT/'person-yolo12m.onnx'),'--output',str(OUTPUT/'linq-person'),
                '--report',str(OUTPUT/'linq-preflight.json')],check=True)

После быстрого продолжения скачайте `linq-fast-trained-models.zip`. В `checkpoint-lineage.json`
сохраняются SHA256 исходных last.pt и число уже завершённых эпох. Люди и каски
сохраняют optimizer и номер эпохи; данные и лимит времени изменены. `resume-last.pt`
сохраняет optimizer для следующего продолжения даже после финального ONNX-экспорта.
Проверьте метрики, полный held-out и реальные видео перед установкой в сайт.
Для LinQ нужны отдельная SDK-компиляция и аппаратная проверка.

In [ ]:
import hashlib
progress=json.loads((OUTPUT/'training-progress.json').read_text())
complete=len(progress['jobs'])==5 and all(j['status']=='done' for j in progress['jobs'].values())
manifest={'mode':'fast','complete':complete,'torch':torch.__version__,
          'gpus':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
          'source_manifest_sha256':hashlib.sha256((FULL_DATA/'manifest.jsonl').read_bytes()).hexdigest(),
          'selection_sha256':hashlib.sha256((DATA/'selection.jsonl').read_bytes()).hexdigest(),
          'trainer_sha256':hashlib.sha256((SCRIPTS/'train_ppe.py').read_bytes()).hexdigest(),
          'test_scope':'fixed bounded subset; full held-out not evaluated',
          'linq_hardware_verified':False,'event_metrics':'not_evaluated',
          'warning':'Short fine-tuning; independent quality must be measured before deployment.'}
(OUTPUT/'run-manifest.json').write_text(json.dumps(manifest,indent=2))
import zipfile
archive=WORK/'linq-fast-trained-models.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for file in OUTPUT.rglob('*'):
        if file.is_file():z.write(file,file.relative_to(OUTPUT))
print('Download:',archive,'All five tasks complete:',complete,flush=True)
assert complete, 'Partial run: inspect saved progress/checkpoints; do not treat missing models as completed.'